# 로컬 음식 사진 배경 교체 검증

학습한 음식 전용 YOLO11n `models/best.pt` → SAM → BiRefNet 품질 검사 → 배경 생성 → OpenCLIP 차단 검증을 로컬에서 확인합니다. 운영 모드에서는 중앙 사각형 대체를 사용하지 않습니다.

In [ ]:
from pathlib import Path
import json, subprocess, sys
from PIL import Image
from IPython.display import display
PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / 'configs/pipeline.yaml').is_file(), 'Jupyter를 food-image-cleanup-pipeline 폴더에서 실행하세요.'
print('프로젝트:', PROJECT_ROOT)
print('Python:', sys.executable)

In [ ]:
# 최초 1회에만 실행합니다.
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-local.txt'], check=True)
# subprocess.run([sys.executable, '-m', 'scripts.download_models', '--models', 'yolo', 'sam2', 'big-lama', 'openclip', 'birefnet', 'sana'], check=True)

DETECTOR_PROFILE = 'food_specialized'  # 비교가 필요하면 coco_yolo11n으로 변경합니다.
DETECTOR_WEIGHTS = PROJECT_ROOT / 'models/best.pt' if DETECTOR_PROFILE == 'food_specialized' else PROJECT_ROOT / 'models/yolo11n.pt'
ANGLE_WEIGHTS = PROJECT_ROOT / 'models/efficientnet_best.pt'
SAM_SMALL_WEIGHTS = PROJECT_ROOT / 'models/sam2.1_s.pt'
assert SAM_SMALL_WEIGHTS.is_file(), f'SAM 2.1 Small 가중치가 없습니다: {SAM_SMALL_WEIGHTS}. 위 모델 다운로드 셀을 다시 실행하세요.'
assert DETECTOR_WEIGHTS.is_file(), f'선택한 탐지기 가중치가 없습니다: {DETECTOR_WEIGHTS}'
assert ANGLE_WEIGHTS.is_file(), f'EfficientNet-B0 각도 분류 가중치가 없습니다: {ANGLE_WEIGHTS}'
print(f'탐지 프로필: {DETECTOR_PROFILE}, 가중치: {DETECTOR_WEIGHTS}')
print(f'각도 분류기: {ANGLE_WEIGHTS}')
print(f'전경 탐지 순서: GroundingDINO 우선 → YOLO 보완, SAM: {SAM_SMALL_WEIGHTS.name}')

In [ ]:
INPUT_PATH = Path('data/input/example.jpg')
assert INPUT_PATH.is_file(), f'입력 사진을 찾을 수 없습니다: {INPUT_PATH}'
display(Image.open(INPUT_PATH))
metadata = {'business_type':'cafe', 'food_category':'dessert', 'foreground_position':'center_lower', 'light_direction':'left'}
METADATA_PATH = PROJECT_ROOT / 'data/input' / f'{INPUT_PATH.stem}_metadata.json'
METADATA_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

In [ ]:
# 운영과 같은 안전 모드입니다. 탐지 실패 또는 OpenCLIP 실패 시 광고 합성 JPG를 만들지 않습니다.
command = [sys.executable, '-m', 'scripts.run_background_replacement', '--input', str(INPUT_PATH), '--metadata', str(METADATA_PATH), '--enable-matting', '--enable-background-generator', '--detector-profile', DETECTOR_PROFILE]
result = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(result.stdout)
if result.stderr: print(result.stderr)
print('종료 코드:', result.returncode)
# 연결 테스트에만 command.append('--diagnostic-center-fallback')을 사용합니다.

In [ ]:
from IPython.display import Image as DisplayImage
report_path = PROJECT_ROOT / 'data/reports' / f'{INPUT_PATH.stem}_background_replacement_report.json'
assert report_path.is_file(), f'보고서가 없습니다: {report_path}'
report = json.loads(report_path.read_text(encoding='utf-8'))
detector_stage = report.get('stages', {}).get('step_2_yolo_detection', {})
angle_stage = report.get('stages', {}).get('step_7_camera_angle_classification', {})
assert detector_stage.get('profile') == DETECTOR_PROFILE, f'탐지 프로필 불일치: {detector_stage}'
if DETECTOR_PROFILE == 'food_specialized':
    assert detector_stage.get('model', '').replace('\\', '/').endswith('models/best.pt'), detector_stage
assert angle_stage.get('model', '').replace('\\', '/').endswith('models/efficientnet_best.pt'), angle_stage
assert angle_stage.get('status') in {'completed', 'low_confidence'}, angle_stage
assert angle_stage.get('label') in {'top', '45'}, angle_stage
print(json.dumps({'status':report.get('status'), 'reason':report.get('reason'), 'detector':detector_stage, 'camera_angle':angle_stage, 'debug_artifacts':report.get('debug_artifacts'), 'validation':report.get('stages',{}).get('step_13_foreground_validation')}, ensure_ascii=False, indent=2))
for name, artifact_path in report.get('debug_artifacts', {}).items():
    artifact = Path(artifact_path)
    if artifact.is_file():
        print(name, artifact)
        display(DisplayImage(filename=str(artifact)))